# 手撕 ORPO (Odds Ratio Preference Optimization)

## 背景
ORPO 将 SFT 和偏好优化合并为一步，无需 reference model。
Loss = SFT loss + λ * Log Odds Ratio loss。
odds ratio = P(chosen) / (1 - P(chosen)) ÷ (P(rejected) / (1 - P(rejected)))

## 考察点
- odds ratio 的定义与直觉
- 无需 reference model 的优势
- SFT + 偏好的一步训练

In [ ]:
import torch
import torch.nn.functional as F

def orpo_loss(policy_chosen_logps, policy_rejected_logps, lambda_ratio=1.0):
    # policy_chosen_logps: log π(y_chosen|x)
    # policy_rejected_logps: log π(y_rejected|x)
    # SFT loss: -log π(y_chosen|x)
    sft_loss = -policy_chosen_logps.mean()
    # Log Odds Ratio: log(odds_chosen / odds_rejected)
    # odds(π) = π / (1 - π) = exp(logp) / (1 - exp(logp))
    # log_odds = logp - log(1 - exp(logp)) = logp - log1p(-exp(logp))
    log_odds_chosen = policy_chosen_logps - torch.log1p(-torch.exp(torch.clamp(policy_chosen_logps, max=-1e-8)))
    log_odds_rejected = policy_rejected_logps - torch.log1p(-torch.exp(policy_rejected_logps))
    log_odds_ratio = log_odds_chosen - log_odds_rejected
    #偏好 loss: -log σ(log_odds_ratio)
    preference_loss = -F.logsigmoid(log_odds_ratio).mean()
    return sft_loss + lambda_ratio * preference_loss

In [ ]:
# 验证 ORPO loss
torch.manual_seed(42)
chosen_logps = torch.randn(8) * 2 - 1  # log probabilities (负值)
rejected_logps = torch.randn(8) * 2 - 1
loss = orpo_loss(chosen_logps, rejected_logps, lambda_ratio=1.0)
assert loss.item() > 0, "loss 应为正"
# 验证：chosen 概率高于 rejected 时 loss 更低
good_chosen = chosen_logps + 1.0  # 提高 chosen 概率
loss_better = orpo_loss(good_chosen, rejected_logps, lambda_ratio=1.0)
assert loss_better < loss, "提高 chosen 概率应降低 loss"
print(f"原始 loss: {loss.item():.4f}")
print(f"提升 chosen 后 loss: {loss_better.item():.4f}")
print("✅ ORPO loss 验证通过")